In [1]:
import requests
import time
import math
import pandas as pd
from pprint import pprint

# ── Specimen under investigation ──────────────────────────────────────────────
SPECIMEN_ID      = "BBM-F-004821"
TAXON_NAME       = "Aureonarius armiae"
COLLECTOR        = "R. Baird"
LATITUDE         = 49.37      # low-precision (2 d.p.)
LONGITUDE        = -123.02
COLLECTION_DATE  = "2021-09-04"
LOCATION_NAME    = "Lynn Headwaters, North Vancouver, BC, Canada"

HEADERS = {
    "User-Agent": "BeatyMuseumCapstone/0.1 (DSCI591; contact=canadasung@gmail.com)"
}

INAT_BASE = "https://api.inaturalist.org/v1"

print(f"Specimen: {SPECIMEN_ID} | Taxon: {TAXON_NAME}")


Specimen: BBM-F-004821 | Taxon: Aureonarius armiae


In [2]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def get_taxon_id(taxon_name: str):
    """Resolve taxon name to iNaturalist taxon id (best match)."""
    url = f"{INAT_BASE}/taxa"
    params = {"q": taxon_name, "per_page": 5}
    r = requests.get(url, params=params, headers=HEADERS)
    r.raise_for_status()
    results = r.json().get("results", [])
    if not results:
        return None, None
    # pick first exact name match if available, else first result
    for rec in results:
        if rec.get("name") == taxon_name:
            return rec.get("id"), rec
    return results[0].get("id"), results[0]

def fetch_observations_for_taxon(taxon_id=None, taxon_name=None,
                                 lat=None, lon=None, radius_km=50,
                                 quality="research", per_page=200, max_pages=5,
                                 sleep_between_pages=0.5):
    """
    Paginate observations. Returns list of raw observation dicts.
    If taxon_id provided, use it; otherwise use taxon_name.
    """
    url = f"{INAT_BASE}/observations"
    params = {
        "per_page": per_page,
        "order_by": "created_at",
        "order": "desc",
        "quality_grade": quality,
        "lat": lat,
        "lng": lon,
        "radius": radius_km,
    }
    if taxon_id:
        params["taxon_id"] = taxon_id
    elif taxon_name:
        params["taxon_name"] = taxon_name

    page = 1
    all_results = []
    while page <= max_pages:
        params["page"] = page
        r = requests.get(url, params=params, headers=HEADERS)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if not results:
            break
        all_results.extend(results)
        # stop if fewer than per_page returned (no more pages)
        if len(results) < per_page:
            break
        page += 1
        time.sleep(sleep_between_pages)
    return all_results

def flatten_observations(obs_list):
    """Return DataFrame with parsed lat/lon and key fields; filter out private/obscured."""
    rows = []
    for o in obs_list:
        # location is "lat,lon" string in v1 API
        loc = o.get("location")
        lat = lon = None
        if loc:
            try:
                lat_str, lon_str = loc.split(",")
                lat = float(lat_str)
                lon = float(lon_str)
            except Exception:
                lat = lon = None
        rows.append({
            "inat_id": o.get("id"),
            "observed_on": o.get("observed_on"),
            "quality_grade": o.get("quality_grade"),
            "taxon_name": (o.get("taxon") or {}).get("name"),
            "community_taxon": (o.get("community_taxon") or {}).get("name"),
            "geoprivacy": o.get("geoprivacy"),
            "latitude": lat,
            "longitude": lon,
            "positional_accuracy": o.get("positional_accuracy"),  # metres
            "observer": (o.get("user") or {}).get("login"),
            "place_guess": o.get("place_guess"),
            "source": "iNaturalist",
            "raw": o
        })
    df = pd.DataFrame(rows)
    # keep only open geoprivacy and non-null coords
    df = df[(df["geoprivacy"].isnull() | (df["geoprivacy"] == "open")) &
            df["latitude"].notnull() & df["longitude"].notnull()]
    # drop exact duplicate coordinates
    df = df.drop_duplicates(subset=["latitude", "longitude"])
    return df.reset_index(drop=True)

def compute_centroid(df):
    """Simple arithmetic mean centroid; returns (lat, lon)."""
    return (df["latitude"].mean(), df["longitude"].mean())

def compute_bbox_center(df):
    """Bounding box center from observations."""
    min_lat, max_lat = df["latitude"].min(), df["latitude"].max()
    min_lon, max_lon = df["longitude"].min(), df["longitude"].max()
    return ((min_lat + max_lat) / 2.0, (min_lon + max_lon) / 2.0)

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lon points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def find_nearest_observation(df, lat, lon):
    """Return row (as dict) of nearest observation to given lat/lon and distance_km."""
    if df.empty:
        return None
    df = df.copy()
    df["dist_km"] = df.apply(lambda r: haversine_km(lat, lon, r["latitude"], r["longitude"]), axis=1)
    nearest = df.sort_values("dist_km").iloc[0]
    return nearest.to_dict()

def round_to_precision(value, precision_digits):
    """Round to given decimal places (e.g., 2 for 2 d.p.)."""
    if value is None or math.isnan(value):
        return value
    return round(value, precision_digits)

In [3]:
# ── Workflow ──────────────────────────────────────────────────────────────────
taxon_id, taxon_record = get_taxon_id(TAXON_NAME)
print("Resolved taxon:", taxon_id, (taxon_record or {}).get("name"))

# fetch observations (increase max_pages if you want more records)
raw_obs = fetch_observations_for_taxon(taxon_id=taxon_id, lat=LATITUDE, lon=LONGITUDE,
                                       radius_km=50, quality="research",
                                       per_page=200, max_pages=5, sleep_between_pages=0.5)
print("Fetched observations:", len(raw_obs))

df_inat = flatten_observations(raw_obs)
print("Filtered usable observations:", len(df_inat))
display_cols = ["inat_id", "observed_on", "taxon_name", "community_taxon",
                "latitude", "longitude", "positional_accuracy", "observer", "place_guess"]
if not df_inat.empty:
    print(df_inat[display_cols].head(10).to_string(index=False))

# compute candidate coordinates
candidates = {}
if not df_inat.empty:
    candidates["centroid"] = compute_centroid(df_inat)
    candidates["bbox_center"] = compute_bbox_center(df_inat)
    nearest = find_nearest_observation(df_inat, LATITUDE, LONGITUDE)
    if nearest:
        candidates["nearest_obs"] = (nearest["latitude"], nearest["longitude"])
        candidates["nearest_obs_id"] = nearest["inat_id"]
        candidates["nearest_obs_dist_km"] = nearest["dist_km"]

# choose final coordinate strategy
# options: "nearest", "centroid", "bbox_center"
STRATEGY = "nearest"  # change as needed

final_coord = None
if STRATEGY == "nearest" and "nearest_obs" in candidates:
    final_coord = candidates["nearest_obs"]
elif STRATEGY == "centroid" and "centroid" in candidates:
    final_coord = candidates["centroid"]
elif STRATEGY == "bbox_center" and "bbox_center" in candidates:
    final_coord = candidates["bbox_center"]

# round to specimen precision (2 decimal places in your example)
PRECISION_DIGITS = 2
if final_coord:
    final_lat = round_to_precision(final_coord[0], PRECISION_DIGITS)
    final_lon = round_to_precision(final_coord[1], PRECISION_DIGITS)
    print("Chosen strategy:", STRATEGY)
    print("Final coordinate (rounded):", final_lat, final_lon)
    # provenance
    provenance = {
        "taxon_id": taxon_id,
        "taxon_name": TAXON_NAME,
        "strategy": STRATEGY,
        "source": "iNaturalist",
        "num_observations_used": len(df_inat),
        "precision_digits": PRECISION_DIGITS
    }
    if "nearest_obs_id" in candidates:
        provenance["nearest_obs_id"] = candidates["nearest_obs_id"]
        provenance["nearest_obs_dist_km"] = candidates["nearest_obs_dist_km"]
    pprint(provenance)
else:
    print("No usable iNaturalist coordinates found for this taxon near the specimen location.")


Resolved taxon: 1433806 Aureonarius armiae
Fetched observations: 0


KeyError: 'geoprivacy'

In [ ]:
# ── 1. Configuration & Constants ──────────────────────────────────────────────
SPECIMEN_ID     = "BBM-F-004821"
ORIGINAL_TAXON  = "Amanita muscaria var. guessowii"
LATITUDE        = 49.37  # Lynn Headwaters
LONGITUDE       = -123.02

# The new target taxon
TARGET_TAXON    = "Aureonarius armiae"

HEADERS = {
    "User-Agent": "BeatyMuseumCapstone/0.1 (DSCI591; contact=canadasung@gmail.com)"
}
INAT_BASE = "https://api.inaturalist.org/v1"

print(f"Specimen: {SPECIMEN_ID} | Original Taxon: {ORIGINAL_TAXON}")
print(f"Targeting new search for: {TARGET_TAXON}\n")

# ── 2. API Helper Functions ───────────────────────────────────────────────────
def get_taxon_id(taxon_name: str) -> int:
    """Resolves a text name to an exact iNaturalist Taxon ID."""
    url = f"{INAT_BASE}/taxa/autocomplete"
    r = requests.get(url, params={"q": taxon_name}, headers=HEADERS)
    r.raise_for_status()
    
    results = r.json().get("results", [])
    if results:
        return results[0]["id"]
    raise ValueError(f"Could not resolve Taxon ID for '{taxon_name}'")

def inat_search_observations(taxon_id: int, lat: float = None, lon: float = None,
                              radius_km: int = 50, place_id: int = None, 
                              quality: str = "research", limit: int = 100) -> dict:
    """Search iNaturalist research-grade observations by coords or place_id."""
    url = f"{INAT_BASE}/observations"
    
    # Base parameters
    params = {
        "taxon_id": taxon_id,
        "quality_grade": quality,
        "per_page": limit,
        "order_by": "created_at",
        "order": "desc",
    }
    
    # Apply location strategy (Coordinates take precedence, fallback to place_id)
    if lat and lon:
        params.update({"lat": lat, "lng": lon, "radius": radius_km})
    elif place_id:
        params.update({"place_id": place_id})
        
    r = requests.get(url, params=params, headers=HEADERS)
    r.raise_for_status()
    return r.json()

# ── 3. Execution ──────────────────────────────────────────────────────────────
try:
    # Resolve the ID first
    target_taxon_id = get_taxon_id(TARGET_TAXON)
    print(f"Resolved '{TARGET_TAXON}' to ID: {target_taxon_id}")
    
    # Search via coordinates (Radius method)
    inat_data = inat_search_observations(
        taxon_id=target_taxon_id, 
        lat=LATITUDE, 
        lon=LONGITUDE, 
        radius_km=50
    )
    
    # Fallback logic: If 0 results in 50km, expand to all of BC (place_id: 7083)
    if inat_data.get('total_results', 0) == 0:
        print(f"0 results found within 50km of Lynn Headwaters. Expanding search to British Columbia...")
        inat_data = inat_search_observations(
            taxon_id=target_taxon_id, 
            place_id=7083  
        )

    print(f"Total matching (research-grade): {inat_data.get('total_results')}")
    print(f"Returned in this page: {len(inat_data.get('results', []))}\n")

except Exception as e:
    print(f"API Error: {e}")
    inat_data = {"results": []}

# ── 4. Flatten iNat observations ──────────────────────────────────────────────
inat_rows = []
for obs in inat_data.get("results", []):
    taxon      = obs.get("taxon") or {}
    community  = obs.get("community_taxon") or {}
    coords     = obs.get("location", "").split(",") if obs.get("location") else [None, None]
    
    inat_rows.append({
        "inat_id":             obs.get("id"),
        "observed_on":         obs.get("observed_on_details", {}).get("date", obs.get("observed_on")),
        "quality_grade":       obs.get("quality_grade"),
        "observer_taxon":      taxon.get("name"),
        "community_taxon":     community.get("name"),
        "identifications_agree": obs.get("identifications_most_agree", True), 
        "geoprivacy":          obs.get("geoprivacy"), # "obscured", "private", or None
        "latitude":            float(coords[0]) if coords[0] else None,
        "longitude":           float(coords[1]) if coords[1] else None,
        "positional_accuracy": obs.get("positional_accuracy"), 
        "observer":            (obs.get("user") or {}).get("login"),
        "place_guess":         obs.get("place_guess"),
        "url":                 obs.get("uri")
    })

df_inat = pd.DataFrame(inat_rows)

# ── 5. Observer-community taxon agreement check ───────────────────────────────
if not df_inat.empty:
    # An observation has dissent if the observer's taxon doesn't match the final community consensus
    disagreements = df_inat[df_inat["observer_taxon"] != df_inat["community_taxon"]]
    
    print(f"Records with observer/community taxon disagreement: {len(disagreements)} / {len(df_inat)}")
    if len(disagreements) > 0:
        print(disagreements[["inat_id", "observer_taxon", "community_taxon", "observer"]])
else:
    print("No data retrieved to analyze.")

### V2 Example

In [ ]:
import requests, time
BASE = "https://api.inaturalist.org/v2"
HEADERS = {"User-Agent": "BeatyMuseumCapstone/0.1 (DSCI591; contact=canadasung@gmail.com)"}

# 1) resolve taxon
r = requests.get(f"{BASE}/taxa", params={"q":"Aureonarius armiae"}, headers=HEADERS)
taxon = r.json()["results"][0]["record"]
taxon_id = taxon["id"]

# 2) fetch observations (basic fields)
params = {"taxon_id": taxon_id, "quality_grade":"research", "per_page":100, "page":1,
          "fields":"id,observed_on,geojson,place_guess,quality_grade,geoprivacy,coordinate_uncertainty"}
obs = []
while True:
    r = requests.get(f"{BASE}/observations", params=params, headers=HEADERS); data = r.json()
    obs.extend(data["results"])
    if not data["pagination"]["has_next"]: break
    params["page"] += 1
    time.sleep(0.2)


In [ ]:
import requests
import pandas as pd

INAT_V2_BASE = "https://api.inaturalist.org/v2"

# ── 1. The Method Override Headers ─────────────────────────────────────────────
HEADERS = {
    "User-Agent": "BeatyMuseumCapstone/0.1 (DSCI591; contact=canadasung@gmail.com)",
    "X-HTTP-Method-Override": "GET",  # Forces the API to treat this POST as a GET
    "Content-Type": "application/json"
}

# ── 2. The Search Criteria (URL Parameters) ──────────────────────────────────
search_params = {
    "iconic_taxa": "Fungi",       # Filter for Fungi
    "quality_grade": "research",  
    "per_page": 10,
    "lat": 49.37,                 # Spatial filtering remains the same
    "lng": -123.02,
    "radius": 50
}

# ── 3. The Fields Payload (JSON Body) ────────────────────────────────────────
# Define exactly what data you want returned. If you don't declare it here, 
# the v2 API will simply drop it from the response to save bandwidth.
payload = {
    "fields": {
        "id": True,
        "observed_on": True,
        "location": True,
        "species_guess": True,
        "identifications_most_agree": True,
        "taxon": {
            "name": True,
            "rank": True
        },
        "user": {
            "login": True
        }
    }
}

# ── 4. Execute Request ───────────────────────────────────────────────────────
# Note the use of requests.post() rather than requests.get()
response = requests.post(
    f"{INAT_V2_BASE}/observations", 
    params=search_params, 
    headers=HEADERS, 
    json=payload
)
response.raise_for_status()
data = response.json()

# ── 5. Flatten to Pandas ─────────────────────────────────────────────────────
fungi_rows = []
for obs in data.get("results", []):
    coords = obs.get("location", "").split(",") if obs.get("location") else [None, None]
    
    fungi_rows.append({
        "inat_id": obs.get("id"),
        "observed_on": obs.get("observed_on"),
        "taxon_name": obs.get("taxon", {}).get("name"),
        "taxon_rank": obs.get("taxon", {}).get("rank"),
        "observer": obs.get("user", {}).get("login"),
        "community_agree": obs.get("identifications_most_agree"),
        "latitude": float(coords[0]) if coords[0] else None,
        "longitude": float(coords[1]) if coords[1] else None,
    })

df_fungi = pd.DataFrame(fungi_rows)
print(f"Retrieved {len(df_fungi)} Fungi records.\n")
print(df_fungi.head())

In [ ]:
import requests
import pandas as pd

INAT_V2_BASE = "https://api.inaturalist.org/v2"

# ── 1. Headers (Method Override for v2) ────────────────────────────────────────
HEADERS = {
    "User-Agent": "BeatyMuseumCapstone/0.1 (DSCI591; contact=canadasung@gmail.com)",
    "X-HTTP-Method-Override": "GET",
    "Content-Type": "application/json"
}

# ── 2. Search Criteria (No Geographic Filters) ─────────────────────────────────
# Notice there is no lat, lng, radius, or place_id. 
# This automatically triggers a global search.
search_params = {
    "taxon_name": "Aureonarius armiae", 
    "quality_grade": "research",
    "per_page": 20
}

# ── 3. Fields Payload ──────────────────────────────────────────────────────────
payload = {
    "fields": {
        "id": True,             # Observation ID: url as /observations/id_value
        "uuid": True,           # Hexa code
        "observed_on": True,
        "location": True,       # The API will still return coordinates if they exist
        "place_guess": True,    # The text description of where it was found
        "species_guess": True,
        "tags": True,
        "locale": True,
        "outlinks": {
            "source": True,
            "url": True
        },
        "taxon": {
            "id": True,                       # Taxon ID
            "parent_id": True,
            "name": True,                     # Taxon name
            "iconic_taxon_id": True,
            "iconic_taxon_name": True,
            "is_active": True,
            "native": True,
            "observations_count": True,
            "wikipedia_url": True
        },
        "user": {
            "id": True,                # user id
            "login": True              # user login name 
        }
    }
}

# payload = {
#     "fields": {
#         "id": True,
#         "observed_on": True,
#         "taxon": {
#             "name": True
#         },
#         # Request the photos
#         "observation_photos": {
#             "photo": {
#                 "url": True,
#                 "license_code": True # Good practice if you plan to display them
#             }
#         }
#     }
# }

payload = {
    "fields": "all"
}

In [19]:

# ── 4. Execute Global Request ──────────────────────────────────────────────────
response = requests.post(
    f"{INAT_V2_BASE}/observations", 
    params=search_params, 
    headers=HEADERS, 
    json=payload
)
response.raise_for_status()
data = response.json()
data


{'total_results': 2,
 'page': 1,
 'per_page': 20,
 'results': [{'id': 13360171,
   'cached_votes_total': 0,
   'captive': False,
   'comments': [],
   'comments_count': 0,
   'community_taxon_id': 1433806,
   'created_at': '2018-06-12T22:00:20+12:00',
   'created_at_details': {'date': '2018-06-12',
    'day': 12,
    'hour': 22,
    'month': 6,
    'week': 24,
    'year': 2018},
   'created_time_zone': 'Pacific/Auckland',
   'description': 'Cortinarius, on soil under hard beech (Fuscospora truncata) Specimen photo DAO_0074',
   'faves': [],
   'faves_count': 0,
   'flags': [],
   'geojson': {'coordinates': [171.5224535491, -42.4036508167],
    'type': 'Point'},
   'geoprivacy': None,
   'ident_taxon_ids': [48460,
    47170,
    47169,
    492000,
    50814,
    1094814,
    47167,
    785517,
    48705,
    1417328,
    1433702,
    1433703,
    1433806],
   'identifications': [{'id': 717828708,
     'body': None,
     'category': 'supporting',
     'created_at': '2025-10-26T21:21:40+0

In [ ]:

# ── 5. Flatten to Pandas ───────────────────────────────────────────────────────
global_rows = []
for obs in data.get("results", []):
    global_rows.append({
        "inat_id": obs.get("id"),
        "observed_on": obs.get("observed_on"),
        "taxon_id": obs.get("taxon", {}).get("id"),
        "taxon_name": obs.get("taxon", {}).get("name"),
        "observer": obs.get("user", {}).get("login"),
        "location_name": obs.get("place_guess"),
        "raw_coordinates": obs.get("location") 
    })

df_global = pd.DataFrame(global_rows)

print(f"Total global records retrieved: {len(df_global)}\n")
print(df_global)